# MLOps Model Deployment Notebook: 30-Day Hospital Readmission Risk Prediction

This notebook shows how to manually deploy the trained **30-day hospital readmission risk prediction** model to an Azure Machine Learning managed online endpoint.

It is designed as **Lesson 03** in the workshop sequence and mirrors the deployment part of the automated MLOps workflow.

### Lesson Objective

By the end of this notebook, you will understand how to:

1. Connect to the Azure ML dev workspace
2. Retrieve the latest registered `readmission-model`
3. Create a managed online endpoint
4. Create a `blue` model deployment
5. Route endpoint traffic to the deployment
6. Invoke the endpoint using `data/sample-request.json`
7. Validate the prediction response
8. Clean up the workshop endpoint

### How this fits in the MLOps flow

- **Lesson 01** explores the data.
- **Lesson 02** trains and logs the model.
- **Lesson 03**, this notebook, manually deploys the trained model.
- **Lesson 04** shows how the same steps are automated through pipelines.

> Workshop note: this notebook should run against the **dev** Azure ML workspace. The production workflow uses promotion, approvals and registry-based deployment; those are explained as context, but are not the main path for this lesson.


## Step 1: Environment Setup & Dependencies

Install the Azure ML SDK if your notebook environment does not already include it.

In Azure ML Studio this may already be available. If running locally, ensure you have authenticated with Azure first:

```bash
az login
```

### Required packages

- `azure-ai-ml`: Azure ML SDK v2
- `azure-identity`: authentication support for `DefaultAzureCredential`


In [ ]:
# Uncomment if the notebook environment does not already have these packages.
# %pip install -q azure-ai-ml azure-identity


## Step 2: Imports & Shared Configuration

This section imports the Azure ML SDK classes and defines the shared values used throughout the notebook.

Update the placeholder values for your workshop dev environment before running the notebook.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import re
import time

from azure.ai.ml import MLClient
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment
from azure.identity import DefaultAzureCredential

# Azure ML workspace configuration.
# TODO: update these values for the workshop dev environment.
SUBSCRIPTION_ID = "<your-subscription-id>"
RESOURCE_GROUP = "rg-readmit-dev"
WORKSPACE_NAME = "<your-dev-aml-workspace>"

# MLOps configuration aligned to the repo.
PROJECT_ROOT = Path.cwd()
MODEL_NAME = "readmission-model"
ENDPOINT_BASE_NAME = "readmission-online"
DEPLOYMENT_NAME = "blue"

# Deployment configuration from mlops/azureml/deploy/online/online-deployment.yml
INFERENCE_ENVIRONMENT = "azureml://registries/azureml/environments/mlflow-py312-inference/versions/36"
INSTANCE_TYPE = "Standard_DS4_v2"
INSTANCE_COUNT = 1


## Step 3: Connect to Azure ML Workspace

This is the manual equivalent of the authentication and workspace selection performed by the deployment workflow.

For the workshop, keep the connection explicit so learners can see exactly which **dev** workspace they are deploying into.


In [ ]:
credential = DefaultAzureCredential(exclude_interactive_browser_credential=False)

ml_client = MLClient(
    credential=credential,
    subscription_id=SUBSCRIPTION_ID,
    resource_group_name=RESOURCE_GROUP,
    workspace_name=WORKSPACE_NAME,
)

workspace = ml_client.workspaces.get(WORKSPACE_NAME)

print(f"Connected to workspace: {workspace.name}")
print(f"Resource group: {RESOURCE_GROUP}")
print(f"Location: {workspace.location}")


## Step 4: Retrieve the Latest Registered Model

Lesson 02 trains the model and the pipeline registration step makes it available as `readmission-model`.

The automated deployment workflow retrieves the latest model version before creating the online deployment. This notebook performs the same operation manually using the Azure ML SDK.


In [ ]:
model = ml_client.models.get(name=MODEL_NAME, label="latest")

print("Model selected for deployment")
print(f"Name:    {model.name}")
print(f"Version: {model.version}")
print(f"ID:      {model.id}")
print(f"Type:    {model.type}")
print(f"Tags:    {model.tags}")


## Step 5: Create a Managed Online Endpoint

The repo defines the endpoint in:

```text
mlops/azureml/deploy/online/online-endpoint.yml
```

with:

```yaml
name: readmission-online
auth_mode: aad_token
```

For a workshop notebook, create a unique endpoint name by adding your initials and a timestamp. This avoids name clashes when several people run the lesson at the same time.


In [ ]:
USER_SUFFIX = "kl"  # TODO: change to your initials or workshop group name
DATE_SUFFIX = datetime.now(timezone.utc).strftime("%m%d%H%M")

endpoint_name = f"{ENDPOINT_BASE_NAME}-{USER_SUFFIX}-{DATE_SUFFIX}".lower()
endpoint_name = re.sub(r"[^a-z0-9-]", "-", endpoint_name)[:32].strip("-")

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Manual workshop endpoint for the readmission model",
    auth_mode="aad_token",
)

endpoint_result = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Endpoint created: {endpoint_result.name}")
print(f"Scoring URI: {endpoint_result.scoring_uri}")


## Step 6: Create the Blue Deployment

The repo defines the deployment in:

```text
mlops/azureml/deploy/online/online-deployment.yml
```

Key values from the deployment YAML are:

- deployment name: `blue`
- inference environment: `mlflow-py312-inference`
- instance type: `Standard_DS4_v2`
- instance count: `1`
- model folder variable: `MLFLOW_MODEL_FOLDER=INPUT_model_path`

This cell creates the same deployment manually.


In [ ]:
blue_deployment = ManagedOnlineDeployment(
    name=DEPLOYMENT_NAME,
    endpoint_name=endpoint_name,
    model=model,
    environment=INFERENCE_ENVIRONMENT,
    instance_type=INSTANCE_TYPE,
    instance_count=INSTANCE_COUNT,
    environment_variables={"MLFLOW_MODEL_FOLDER": "INPUT_model_path"},
)

deployment_result = ml_client.online_deployments.begin_create_or_update(blue_deployment).result()

print(f"Deployment created: {deployment_result.name}")
print(f"Provisioning state: {deployment_result.provisioning_state}")


## Step 7: Route Traffic to the Blue Deployment

Creating a deployment does not automatically route traffic to it.

The automated workflow routes endpoint traffic to `blue`. This cell performs that traffic update manually.


In [ ]:
endpoint = ml_client.online_endpoints.get(endpoint_name)
endpoint.traffic = {DEPLOYMENT_NAME: 100}

endpoint_result = ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Traffic routing for {endpoint_result.name}: {endpoint_result.traffic}")


## Step 8: Invoke Endpoint with Sample Request

The repo includes a sample request file at:

```text
data/sample-request.json
```

This is the same request shape used by the smoke test in the deployment workflow.

The notebook first looks for the repo file. If it cannot find it, it writes a fallback sample request using the same feature shape.


In [ ]:
repo_sample_candidates = [
    PROJECT_ROOT / "data" / "sample-request.json",
    PROJECT_ROOT.parent / "data" / "sample-request.json",
]

request_path = next((path for path in repo_sample_candidates if path.exists()), None)

if request_path is None:
    sample_request = {
        "input_data": {
            "columns": [
                "age", "num_prior_admissions", "length_of_stay_days", "num_diagnoses",
                "num_procedures", "num_medications", "num_lab_results",
                "days_since_last_admission", "bmi", "heart_rate_avg",
                "systolic_bp_avg", "hba1c",
                "gender_M",
                "admission_type_Emergency", "admission_type_Trauma", "admission_type_Urgent",
                "discharge_disposition_Home", "discharge_disposition_Home_Health",
                "discharge_disposition_Rehab", "discharge_disposition_SNF",
                "primary_diagnosis_group_Digestive", "primary_diagnosis_group_Endocrine",
                "primary_diagnosis_group_Injury", "primary_diagnosis_group_Musculoskeletal",
                "primary_diagnosis_group_Other", "primary_diagnosis_group_Respiratory",
                "payer_code_Medicare", "payer_code_Other", "payer_code_Private", "payer_code_Self_Pay",
            ],
            "index": [0],
            "data": [[
                72, 3, 8, 6, 2, 12, 15, 45, 29.3, 88, 142, 7.5,
                1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
            ]],
        }
    }

    request_path = PROJECT_ROOT / "sample-request-lesson03.json"
    request_path.write_text(json.dumps(sample_request, indent=2))
    print(f"Repo sample request not found. Wrote fallback request file: {request_path}")
else:
    print(f"Using repo sample request: {request_path}")

response = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(request_path),
)

print("Raw response:")
print(response)


## Step 9: Validate Response & Basic Latency

The workflow smoke test checks that endpoint invocation returns a valid prediction response.

Azure ML / MLflow responses can appear in slightly different shapes, for example:

- `{"predictions": [0]}`
- `[0]`
- a scalar numeric response

This validation confirms the endpoint is returning a prediction rather than only responding with HTTP success.


In [ ]:
parsed = json.loads(response) if isinstance(response, str) else response

# Azure ML / MLflow can sometimes double-encode responses.
if isinstance(parsed, str):
    parsed = json.loads(parsed)

if isinstance(parsed, dict):
    assert "predictions" in parsed, f"Unexpected response dictionary: {parsed}"
elif isinstance(parsed, list):
    assert len(parsed) > 0, "Empty predictions list"
else:
    assert isinstance(parsed, (int, float)), f"Unexpected response type: {type(parsed)}"

print("Integration check PASSED")
print(parsed)

# Optional latency check. Run after the first successful invocation to avoid cold-start effects.
start = time.perf_counter()
_ = ml_client.online_endpoints.invoke(
    endpoint_name=endpoint_name,
    deployment_name=DEPLOYMENT_NAME,
    request_file=str(request_path),
)
latency_ms = (time.perf_counter() - start) * 1000

print(f"Latency: {latency_ms:.0f} ms")
assert latency_ms < 5000, f"Latency {latency_ms:.0f} ms exceeds 5000 ms threshold"


## Step 10: Troubleshooting Checks

Use these cells if endpoint creation, deployment creation or invocation fails.

### Common issues

| Symptom | Likely cause | Fix |
|---|---|---|
| Model not found | Lesson 02 or training workflow has not registered `readmission-model` | Run training and registration first |
| Endpoint name unavailable | Endpoint name collision in the Azure region | Change `USER_SUFFIX` and rerun from Step 5 |
| Deployment failed | Environment/model incompatibility or quota issue | Check deployment logs and vCPU quota |
| Invoke returns auth error | User identity does not have workspace access | Confirm access and Azure sign-in |
| Registry model cannot be pulled in prod | Endpoint identity lacks registry/storage/ACR permissions | Check registry and managed identity role assignments |


In [ ]:
# Check endpoint state
endpoint = ml_client.online_endpoints.get(endpoint_name)
print(endpoint)

# Check deployment state and logs
blue = ml_client.online_deployments.get(name=DEPLOYMENT_NAME, endpoint_name=endpoint_name)
print(blue)

logs = ml_client.online_deployments.get_logs(
    name=DEPLOYMENT_NAME,
    endpoint_name=endpoint_name,
    lines=100,
)
print(logs)


## Step 11: Clean Up Workshop Endpoint

Managed online endpoints consume compute while running.

Delete the endpoint at the end of the lesson unless you intentionally need to keep it for a demo or follow-up test.


In [ ]:
DELETE_ENDPOINT = False

if DELETE_ENDPOINT:
    ml_client.online_endpoints.begin_delete(name=endpoint_name).result()
    print(f"Deleted endpoint: {endpoint_name}")
else:
    print(f"Endpoint left running: {endpoint_name}")
    print("Set DELETE_ENDPOINT = True and re-run this cell to clean up.")


## Summary: Deploy Step in the Full MLOps Pipeline

### What We Completed (This Notebook)

**Deployment Flow:**

1. Connected to the Azure ML dev workspace
2. Retrieved the latest registered `readmission-model`
3. Created a managed online endpoint
4. Created the `blue` deployment
5. Routed 100% traffic to `blue`
6. Invoked the endpoint using `data/sample-request.json`
7. Validated the prediction response and latency
8. Included troubleshooting and cleanup steps

### How this maps back to automation

| Manual notebook step | Automated workflow step |
|---|---|
| Connect to dev workspace | Azure login and workspace resolution |
| Retrieve latest model | `az ml model show --name readmission-model --label latest` |
| Create endpoint | `az ml online-endpoint create` |
| Create deployment | `az ml online-deployment create` or update |
| Route traffic | `az ml online-endpoint update --traffic blue=100` |
| Invoke endpoint | `az ml online-endpoint invoke --request-file data/sample-request.json` |
| Validate response | Integration smoke test assertions |
| Clean up | Manual workshop cleanup or lifecycle automation |

### Production Deployment Context

In the full multi-environment pipeline:

1. A model is trained and evaluated.
2. Passing models are registered.
3. Approved models are promoted through environments.
4. Production deployment uses the shared registry and controlled permissions.
5. Approval gates help prevent unreviewed promotion.

### Key Teaching Point

CI/CD is not doing anything magic. It is executing the same logical deployment steps consistently, with approvals, environment isolation and repeatable configuration.
